## Author: Kelly Tan Jie Li
---

In [ ]:
from pyspark.sql.functions import *
from classes.part_of_speech import FindPartOfSpeech
from pyspark.sql import SparkSession
import subprocess

In [2]:
# Delete old HDFS version
subprocess.run(['hdfs', 'dfs', '-rm','dictionary1/*'])

rm: `dictionary1/*': No such file or directory


CompletedProcess(args=['hdfs', 'dfs', '-rm', 'dictionary1/*'], returncode=1)

In [ ]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("Dictionary Processor") \
    .getOrCreate()

# File paths and base URL
input_file = 'dictionary/dictionary.csv'
base_url = 'https://prpm.dbp.gov.my/Cari1?keyword='
output_dir = '/home/student/de-assgt/content/'
file_name = 'dictionary1.csv'
hdfs_path = 'dictionary1'

# Process the dictionary CSV
final_df = FindPartOfSpeech.process_pos(input_file, base_url, output_dir, file_name, spark)

In [ ]:
# Save DataFrame to HDFS
final_df.coalesce(1).write.csv(hdfs_path, header=True, mode="overwrite")
print(f"Updated dictionary saved to local file and HDFS at: {hdfs_path}")

In [ ]:
# Rename part-00000 file
subprocess.run(['hdfs', 'dfs', '-mv', 'dictionary1/part-00000*', 'dictionary1/dictionary1.csv'])

In [ ]:
input_path = 'dictionary1/dictionary1.csv'
df = spark.read.csv(input_path, header=True, inferSchema=True)
df.show()

# Word Frequency Analysis
word_frequency = df.groupBy("words").agg(count("*").alias("frequency"))

# Most common words
most_common_words = word_frequency.orderBy(desc("frequency")).limit(3)
print("Most Common Words:")
most_common_words.show(truncate=False)

# Least common words
least_common_words = word_frequency.orderBy("frequency").limit(3)
print("Least Common Words:")
least_common_words.show(truncate=False)

# Part-of-Speech Distribution
pos_counts = df.groupBy("pos_tag").agg(count("*").alias("count"))
print("Part-of-Speech Distribution:")
pos_counts.orderBy(desc("count")).show(truncate=False)

# Exclude '-' from POS tag analysis
filtered_pos_counts = pos_counts.filter(col("pos_tag") != "-")

# Most common POS tag 
most_common_pos_tag = filtered_pos_counts.orderBy(desc("count")).limit(1)
print("Most Common POS Tag (excluding '-'):")
most_common_pos_tag.show(truncate=False)

# Least common POS tag 
least_common_pos_tag = filtered_pos_counts.orderBy("count").limit(1)
print("Least Common POS Tag (excluding '-'):")
least_common_pos_tag.show(truncate=False)


In [ ]:
spark.stop()